In [1]:
from glob import glob
import pandas as pd
import random
import stanza
# stanza.download("pt")
nlp = stanza.Pipeline("pt", download_method=None)

2026-06-06 14:36:08 INFO: Loading these models for language: pt (Portuguese):
| Processor    | Package         |
----------------------------------
| tokenize     | bosque          |
| mwt          | bosque          |
| pos          | bosque_charlm   |
| lemma        | bosque_nocharlm |
| constituency | cintil_charlm   |
| depparse     | bosque_charlm   |

2026-06-06 14:36:08 INFO: Using device: cuda
2026-06-06 14:36:08 INFO: Loading: tokenize
2026-06-06 14:36:10 INFO: Loading: mwt
2026-06-06 14:36:10 INFO: Loading: pos
2026-06-06 14:36:13 INFO: Loading: lemma
2026-06-06 14:36:13 INFO: Loading: constituency
2026-06-06 14:36:14 INFO: Loading: depparse
2026-06-06 14:36:15 INFO: Done loading processors!


In [17]:
verbs_file = "./verbs.csv"
verbs_df = pd.read_csv(verbs_file)

contractions_file = "./contraction.csv"
contractions_df = pd.read_csv(contractions_file)

In [42]:
def process_contraction(token):
    text1 = token.words[0].text
    lemma1 = token.words[0].lemma
    
    candidates = contractions_df.loc[
        (contractions_df["lemma1"] == lemma1) &
        (contractions_df["text1"] != text1),
        "text"
    ].dropna().tolist()

    print(candidates)

    if not candidates:
        return token.text 
    
    rng = random.Random()
    new_word = rng.choice(candidates)
    return new_word

def process_verb(word):
    text = word.text
    lemma = word.lemma

    candidates = verbs_df.loc[
        (verbs_df["lemma"] == lemma) &
        (verbs_df["text"] != text),
        "text"
    ].dropna().tolist()

    if not candidates:
        return text

    rng = random.Random()
    new_word = rng.choice(candidates)
    return new_word

In [43]:
def process_text(text):
    new_text = ""

    doc = nlp(text)
    print(text)
    for sent in doc.sentences:
        for token in sent.tokens:
            words = token.words
            if len(words) > 1:
                new_contraction = process_contraction(token)
                new_text += new_contraction + " "

            elif words[0].pos == "VERB":
                verb = process_verb(words[0])
                new_text += verb + " "

            else:
                new_text += words[0].text + " "

    return new_text

In [45]:
process_text("Prezado Sr. Ministro!\nNo momento em que redijo estas linhas, năo conheço o nome de quem foi escolhido para a pasta da Educaçăo. Isto, porém, pouco importa. Năo escrevo para aprovar ou criticar quem ocupará esse cargo no novo governo.\nEsta carta foi escrita para o senhor, seja o senhor quem for, com o intuito de dizer-lhe que - venha de que partido vier, tenha sido de alguma universidade reitor ou diretor - um atributo, sim, năo poderá lhe faltar: o de ter pela educaçăo um profundo, um radical amor.\nNăo serei seu assessor, Sr. Ministro, mas exerço uma funçăo dentro do seu ministério muito mais importante que qualquer outra. Eu sou, talvez também como o senhor, um professor.\nE todos os problemas da educaçăo oferecida em âmbito municipal, estadual ou federal (em instituiçőes públicas ou particulares) săo problemas que, se (é bem verdade) referem-se a questőes de ordem sócio-econômico-familiar, no próprio âmbito escolar radicam e, em particular, na figura, na pessoa de cada professor.")

Prezado Sr. Ministro!
No momento em que redijo estas linhas, năo conheço o nome de quem foi escolhido para a pasta da Educaçăo. Isto, porém, pouco importa. Năo escrevo para aprovar ou criticar quem ocupará esse cargo no novo governo.
Esta carta foi escrita para o senhor, seja o senhor quem for, com o intuito de dizer-lhe que - venha de que partido vier, tenha sido de alguma universidade reitor ou diretor - um atributo, sim, năo poderá lhe faltar: o de ter pela educaçăo um profundo, um radical amor.
Năo serei seu assessor, Sr. Ministro, mas exerço uma funçăo dentro do seu ministério muito mais importante que qualquer outra. Eu sou, talvez também como o senhor, um professor.
E todos os problemas da educaçăo oferecida em âmbito municipal, estadual ou federal (em instituiçőes públicas ou particulares) săo problemas que, se (é bem verdade) referem-se a questőes de ordem sócio-econômico-familiar, no próprio âmbito escolar radicam e, em particular, na figura, na pessoa de cada professor.
['na

'Prezado Sr. Ministro ! nutre momento em que redijo estas linhas , năo conhecia o nome de quem foi escolher para a pasta Daí Educaçăo . Isto , porém , pouco importadas . Năo escrevesse para aprovadas ou Criticou quem ocupam esse cargo Nestas novo governo . Esta carta foi Escrevemos para o senhor , seja o senhor quem irem , com o intuito de Diz-lhe que - vir de que partido visse , tenha sido de alguma universidade reitor ou diretor - um atributo , sim , năo Pode lhe faltará : o de terão Pelo educaçăo um profundo , um radical amor . Năo serei seu assessor , Sr. Ministro , mas exerço uma funçăo dentro Dos seu ministério muito mais importante que qualquer outra . Eu sou , talvez também como o senhor , um professor . E todos os problemas DA educaçăo oferecera em âmbito municipal , estadual ou federal ( em instituiçőes públicas ou particulares ) sonham problemas que , se ( é bem verdade ) referir-se a questőes de ordem sócio-econômico-familiar , Nesse próprio âmbito escolar radicada e , em p